# 01 -- Download and explore raw data

Download the Home Credit Default Risk tables from Kaggle into `data/raw/`,
then inspect file sizes, shapes, columns, and sample rows.

Exploration is **memory-light**: we do not keep every table in RAM at once.
Large files are scanned in chunks (row counts, missingness, join keys).
Only `application_train.csv` is loaded fully for target checks.

Requires:
- Kaggle account with **competition rules accepted** (one-time, in browser)
- `~/.kaggle/kaggle.json` credentials
- `kaggle` CLI on PATH

If download returns **403 Forbidden**, open:
https://www.kaggle.com/competitions/home-credit-default-risk/rules
and click **I Understand and Accept** / Join Competition, then re-run the download cell.

If you already hit a `MemoryError`, **restart the kernel** before running this notebook.

In [1]:
from __future__ import annotations

import gc
import subprocess
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DAYS_EMPLOYED_ANOMALY, RAW_DATA_DIR, RAW_TABLE_FILES

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
COMPETITION = "home-credit-default-risk"
ZIP_NAME = f"{COMPETITION}.zip"
ZIP_PATH = RAW_DATA_DIR / ZIP_NAME
CHUNKSIZE = 500_000
SAMPLE_ROWS = 50_000

PROJECT_ROOT, RAW_DATA_DIR, CHUNKSIZE, SAMPLE_ROWS

(WindowsPath('C:/Users/user/Desktop/Credit-Scoring Project'),
 WindowsPath('C:/Users/user/Desktop/Credit-Scoring Project/data/raw'),
 500000,
 50000)

## Download

Exact CLI equivalent:

```bash
kaggle competitions download -c home-credit-default-risk -p data/raw
```

Then unzip any `.zip` files into `data/raw/` so these CSVs exist:
`application_train.csv`, `bureau.csv`, `bureau_balance.csv`,
`previous_application.csv`, `installments_payments.csv`,
`POS_CASH_balance.csv`, `credit_card_balance.csv`.

Skip download if all expected CSVs are already present.

In [2]:
def expected_csvs_present(raw_dir: Path) -> bool:
    return all((raw_dir / name).exists() for name in RAW_TABLE_FILES)


def download_competition(raw_dir: Path, zip_path: Path) -> None:
    """Download competition zip via Kaggle CLI."""
    cmd = [
        "kaggle",
        "competitions",
        "download",
        "-c",
        COMPETITION,
        "-p",
        str(raw_dir),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if not zip_path.exists():
        print("Zip not found at", zip_path)
        print("Contents of raw dir:", sorted(p.name for p in raw_dir.iterdir()))


def unzip_archives(raw_dir: Path) -> list[Path]:
    """Extract any .zip files under raw_dir into raw_dir."""
    extracted: list[Path] = []
    for zpath in sorted(raw_dir.glob("*.zip")):
        print(f"Extracting {zpath.name} ...")
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(raw_dir)
        extracted.append(zpath)
    return extracted


if expected_csvs_present(RAW_DATA_DIR):
    print("All expected CSVs already present. Skipping download.")
else:
    if not any(RAW_DATA_DIR.glob("*.zip")) and not expected_csvs_present(RAW_DATA_DIR):
        download_competition(RAW_DATA_DIR, ZIP_PATH)
    unzip_archives(RAW_DATA_DIR)

missing = [f for f in RAW_TABLE_FILES if not (RAW_DATA_DIR / f).exists()]
print("Missing expected tables:" if missing else "All expected tables present.")
missing

All expected CSVs already present. Skipping download.
All expected tables present.


[]

## What is on disk?

In [3]:
rows = []
for path in sorted(RAW_DATA_DIR.iterdir()):
    if path.name == ".gitkeep":
        continue
    size_mb = path.stat().st_size / (1024 ** 2)
    rows.append(
        {
            "file": path.name,
            "type": path.suffix.lower() or "dir",
            "size_mb": round(size_mb, 2),
            "expected_table": path.name in RAW_TABLE_FILES,
        }
    )

files_df = pd.DataFrame(rows).sort_values("size_mb", ascending=False)
files_df

,file,type,size_mb,expected_table
7,installments_payments.csv,.csv,689.62,True
5,home-credit-default-risk.zip,.zip,688.19,False
4,credit_card_balance.csv,.csv,404.91,True
9,previous_application.csv,.csv,386.21,True
8,POS_CASH_balance.csv,.csv,374.51,True
3,bureau_balance.csv,.csv,358.19,True
2,bureau.csv,.csv,162.14,True
1,application_train.csv,.csv,158.44,True
0,application_test.csv,.csv,25.34,False
10,sample_submission.csv,.csv,0.51,False


## Memory-light table summary

For each expected CSV:
- column names from the header only
- exact row count and null counts via chunked reads
- a small sample for `head()` / dtype preview

Nothing large is retained after each table is scanned.

In [4]:
def scan_csv(path: Path, chunksize: int = CHUNKSIZE) -> dict:
    """Chunked profile: rows, cols, nulls. Does not keep the full frame."""
    header = pd.read_csv(path, nrows=0)
    cols = list(header.columns)
    n_rows = 0
    null_counts = pd.Series(0, index=cols, dtype="int64")

    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False):
        n_rows += len(chunk)
        null_counts = null_counts.add(chunk.isna().sum(), fill_value=0).astype("int64")
        del chunk

    gc.collect()
    return {
        "table": path.name,
        "status": "ok",
        "rows": int(n_rows),
        "cols": len(cols),
        "columns": cols,
        "null_counts": null_counts,
    }


summaries = []
column_profiles: dict[str, pd.DataFrame] = {}

for name in RAW_TABLE_FILES:
    path = RAW_DATA_DIR / name
    if not path.exists():
        summaries.append({"table": name, "status": "missing"})
        continue

    print(f"Scanning {name} ...")
    info = scan_csv(path)
    summaries.append(
        {
            "table": info["table"],
            "status": info["status"],
            "rows": info["rows"],
            "cols": info["cols"],
            "pct_cols_with_nulls": round(
                100 * (info["null_counts"] > 0).mean(), 2
            ),
        }
    )

    sample = pd.read_csv(path, nrows=SAMPLE_ROWS, low_memory=False)
    overview = pd.DataFrame(
        {
            "dtype": sample.dtypes.astype(str),
            "n_unique_in_sample": sample.nunique(dropna=True),
            "pct_missing_full": (info["null_counts"] / max(info["rows"], 1) * 100).round(2),
        }
    ).sort_values("pct_missing_full", ascending=False)
    column_profiles[name] = overview

    print(f"  rows={info['rows']:,} cols={info['cols']}")
    display(overview.head(12))
    display(sample.head(3))
    del sample, info
    gc.collect()

summary_df = pd.DataFrame(summaries)
summary_df

Scanning application_train.csv ...
  rows=307,511 cols=122


,dtype,n_unique_in_sample,pct_missing_full
COMMONAREA_AVG,float64,1956,69.87
COMMONAREA_MODE,float64,1925,69.87
COMMONAREA_MEDI,float64,1973,69.87
NONLIVINGAPARTMENTS_MEDI,float64,128,69.43
NONLIVINGAPARTMENTS_MODE,float64,100,69.43
NONLIVINGAPARTMENTS_AVG,float64,219,69.43
FONDKAPREMONT_MODE,object,4,68.39
LIVINGAPARTMENTS_AVG,float64,1130,68.35
LIVINGAPARTMENTS_MEDI,float64,807,68.35
LIVINGAPARTMENTS_MODE,float64,580,68.35


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


Scanning bureau.csv ...
  rows=1,716,428 cols=17


,dtype,n_unique_in_sample,pct_missing_full
AMT_ANNUITY,float64,3103,71.47
AMT_CREDIT_MAX_OVERDUE,float64,3348,65.51
DAYS_ENDDATE_FACT,float64,2764,36.92
AMT_CREDIT_SUM_LIMIT,float64,1966,34.48
AMT_CREDIT_SUM_DEBT,float64,12622,15.01
DAYS_CREDIT_ENDDATE,float64,6011,6.15
SK_ID_CURR,int64,10171,0.00
DAYS_CREDIT,int64,2920,0.00
SK_ID_BUREAU,int64,50000,0.00
CREDIT_DAY_OVERDUE,int64,95,0.00


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN


... (12 outputs left out)

Scanning credit_card_balance.csv ...
  rows=3,840,312 cols=23


,dtype,n_unique_in_sample,pct_missing_full
AMT_PAYMENT_CURRENT,float64,10540,20.00
CNT_DRAWINGS_POS_CURRENT,float64,57,19.52
AMT_DRAWINGS_ATM_CURRENT,float64,330,19.52
CNT_DRAWINGS_ATM_CURRENT,float64,20,19.52
AMT_DRAWINGS_POS_CURRENT,float64,2504,19.52
AMT_DRAWINGS_OTHER_CURRENT,float64,100,19.52
CNT_DRAWINGS_OTHER_CURRENT,float64,6,19.52
CNT_INSTALMENT_MATURE_CUM,float64,114,7.95
AMT_INST_MIN_REGULARITY,float64,6102,7.95
AMT_DRAWINGS_CURRENT,float64,2930,0.00


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0


,table,status,rows,cols,pct_cols_with_nulls
0,application_train.csv,ok,307511,122,54.92
1,bureau.csv,ok,1716428,17,41.18
2,bureau_balance.csv,ok,27299925,3,0.00
3,previous_application.csv,ok,1670214,37,43.24
4,installments_payments.csv,ok,13605401,8,25.00
5,POS_CASH_balance.csv,ok,10001358,8,25.00
6,credit_card_balance.csv,ok,3840312,23,39.13


## Application train: target and keys

`application_train.csv` is small enough to load fully (~300k rows).

In [5]:
app_path = RAW_DATA_DIR / "application_train.csv"
app = pd.read_csv(app_path, low_memory=False)

print("shape:", app.shape)
print("TARGET value counts:")
display(app["TARGET"].value_counts(dropna=False))
print("TARGET rate:", round(app["TARGET"].mean(), 4))
print("Unique SK_ID_CURR:", app["SK_ID_CURR"].nunique(), "rows:", len(app))
print("Duplicate SK_ID_CURR:", int(app["SK_ID_CURR"].duplicated().sum()))

n_anom = int((app["DAYS_EMPLOYED"] == DAYS_EMPLOYED_ANOMALY).sum())
print(
    f"DAYS_EMPLOYED == {DAYS_EMPLOYED_ANOMALY}: {n_anom} rows "
    f"({100 * n_anom / len(app):.2f}%)"
)
display(app["DAYS_EMPLOYED"].describe())

shape: (307511, 122)
TARGET value counts:


TARGET
0    282686
1     24825
Name: count, dtype: int64

TARGET rate: 0.0807
Unique SK_ID_CURR: 307511 rows: 307511
Duplicate SK_ID_CURR: 0
DAYS_EMPLOYED == 365243: 55374 rows (18.01%)


count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64

## Join-key coverage vs application_train

Only the ID columns are read, in chunks, so large child tables stay off the heap.

In [6]:
def unique_ids_from_csv(path: Path, id_col: str, chunksize: int = CHUNKSIZE) -> set[int]:
    ids: set[int] = set()
    for chunk in pd.read_csv(path, usecols=[id_col], chunksize=chunksize):
        ids.update(chunk[id_col].dropna().astype("int64").tolist())
        del chunk
    gc.collect()
    return ids


app_ids = set(app["SK_ID_CURR"].astype("int64"))
coverage_rows = []

id_tables = {
    "bureau.csv": "SK_ID_CURR",
    "previous_application.csv": "SK_ID_CURR",
    "installments_payments.csv": "SK_ID_CURR",
    "POS_CASH_balance.csv": "SK_ID_CURR",
    "credit_card_balance.csv": "SK_ID_CURR",
}

for table_name, id_col in id_tables.items():
    path = RAW_DATA_DIR / table_name
    if not path.exists():
        coverage_rows.append({"table": table_name, "status": "missing"})
        continue
    print(f"Collecting {id_col} from {table_name} ...")
    child_ids = unique_ids_from_csv(path, id_col)
    overlap = app_ids & child_ids
    coverage_rows.append(
        {
            "table": table_name,
            "status": "ok",
            "n_ids_in_child": len(child_ids),
            "n_overlap_with_app": len(overlap),
            "pct_app_with_child": round(100 * len(overlap) / len(app_ids), 2),
        }
    )
    del child_ids, overlap
    gc.collect()

# bureau_balance joins via SK_ID_BUREAU
bureau_path = RAW_DATA_DIR / "bureau.csv"
bureau_bal_path = RAW_DATA_DIR / "bureau_balance.csv"
if bureau_path.exists() and bureau_bal_path.exists():
    print("Collecting SK_ID_BUREAU from bureau / bureau_balance ...")
    bur_ids = unique_ids_from_csv(bureau_path, "SK_ID_BUREAU")
    bal_ids = unique_ids_from_csv(bureau_bal_path, "SK_ID_BUREAU")
    coverage_rows.append(
        {
            "table": "bureau_balance.csv (via SK_ID_BUREAU)",
            "status": "ok",
            "n_ids_in_child": len(bal_ids),
            "n_overlap_with_app": len(bur_ids & bal_ids),
            "pct_app_with_child": round(
                100 * len(bur_ids & bal_ids) / max(len(bur_ids), 1), 2
            ),
        }
    )
    del bur_ids, bal_ids
    gc.collect()

coverage_df = pd.DataFrame(coverage_rows)
coverage_df

,table,status,n_ids_in_child,n_overlap_with_app,pct_app_with_child
0,bureau.csv,ok,305811,263491,85.69
1,previous_application.csv,ok,338857,291057,94.65
2,installments_payments.csv,ok,339587,291643,94.84
3,POS_CASH_balance.csv,ok,337252,289444,94.12
4,credit_card_balance.csv,ok,103558,86905,28.26
5,bureau_balance.csv (via SK_ID_BUREAU),ok,817395,774354,45.11


## Optional: peek one large table by name

Change `PEEK_TABLE` and re-run. Uses a sample only.

In [7]:
PEEK_TABLE = "installments_payments.csv"

peek_path = RAW_DATA_DIR / PEEK_TABLE
peek = pd.read_csv(peek_path, nrows=SAMPLE_ROWS, low_memory=False)
print(PEEK_TABLE, "sample shape:", peek.shape)
display(peek.dtypes.astype(str).to_frame("dtype"))
display(peek.head())
display(peek.describe(include="all").T.head(20))
del peek
gc.collect()

installments_payments.csv sample shape: (50000, 8)


,dtype
SK_ID_PREV,int64
SK_ID_CURR,int64
NUM_INSTALMENT_VERSION,float64
NUM_INSTALMENT_NUMBER,int64
DAYS_INSTALMENT,float64
DAYS_ENTRY_PAYMENT,float64
AMT_INSTALMENT,float64
AMT_PAYMENT,float64


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


,count,mean,std,min,25%,50%,75%,max
SK_ID_PREV,50000.0,1.898847e+06,535074.608843,1000011.0,1.430020e+06,1891193.000,2.357910e+06,2843477.00
SK_ID_CURR,50000.0,1.502133e+05,28930.191282,100003.0,1.252740e+05,150360.500,1.752890e+05,199999.00
NUM_INSTALMENT_VERSION,50000.0,8.524400e-01,1.018473,0.0,0.000000e+00,1.000,1.000000e+00,34.00
NUM_INSTALMENT_NUMBER,50000.0,1.900488e+01,26.955871,1.0,4.000000e+00,8.000,1.900000e+01,216.00
DAYS_INSTALMENT,50000.0,-1.046855e+03,802.562359,-2922.0,-1.664000e+03,-828.000,-3.600000e+02,-2.00
DAYS_ENTRY_PAYMENT,50000.0,-1.055467e+03,802.127365,-2989.0,-1.671250e+03,-837.000,-3.690000e+02,-2.00
AMT_INSTALMENT,50000.0,1.742439e+04,56656.449066,0.0,4.181962e+03,8842.905,1.664176e+04,2292736.86
AMT_PAYMENT,50000.0,1.764522e+04,61280.637152,0.0,3.375000e+03,8095.050,1.596104e+04,2292736.86


0

## Notes for next steps

- Raw files live under `data/raw/` (gitignored).
- You can delete `data/raw/home-credit-default-risk.zip` after extract to free disk.
- Next notebook: deeper data-quality checks before feature SQL.
- Do not use positive relative-day offsets as application-time features.
- Later pipeline stages should prefer DuckDB / chunked IO over loading all CSVs into pandas.